Notebook that dumps all Wikipedia articles with their titles into an sqlite3 database

In [ ]:
# Create sqlite3 DB tables if notexistent

import sqlite3

connection = sqlite3.connect("wikipedia.db")
cursor = connection.cursor()
cursor.execute("CREATE VIRTUAL TABLE IF NOT EXISTS articles USING fts5(title, content)")
connection.commit()

In [ ]:
# Define function that extracts the pages from the XML dump
import xml.etree.ElementTree as ET
NAMESPACE="{http://www.mediawiki.org/xml/export-0.10/}"

def extract_pages(xml_path):
    with open(xml_path, "rb") as f:
        context = ET.iterparse(f, events=("end",))
        for _, element in context:
            if element.tag == f"{NAMESPACE}page":
                title = element.findtext(f"{NAMESPACE}title")
                text = element.findtext(f"{NAMESPACE}revision/{NAMESPACE}text")
                
                # Check if title and text are present in a given page
                if title and text:
                    # Check if site is a redirect
                    if not text.lstrip().upper().startswith("#REDIRECT"):
                        # Give title and text to calling function
                        yield title, text
                
                element.clear()

In [ ]:
# Extract data in batches

batch_size = 5000
current_batch = []

# Define function to import data
def data_import(batch):
    cursor.executemany("INSERT INTO articles (title, content) VALUES (?, ?)", batch)
    connection.commit()

# Iterate over data
for i, (title, text) in enumerate(extract_pages("enwiki-20210101-pages-articles-multistream.xml")):
    # Append data to batch
    current_batch.append((title, text))

    # Insert batch into DB
    if len(current_batch) >= batch_size:
        data_import(current_batch)

        # Clean batch
        current_batch = []
        print(f"Inserted {i+1} articles so far")

# Insert remaining data
if current_batch:
    data_import(current_batch)

connection.close()